# ASR adversarial attacks tutorial

Install the package first (`pip install -e .` from the repo root). This notebook uses torchaudio wav2vec2. GPU recommended.

In [ ]:
import torch
import torchaudio
from IPython.display import Audio

from asr_attacks import ASRAttacker, CTCModuleBackend

In [ ]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()
device = "cuda" if torch.cuda.is_available() else "cpu"
backend = CTCModuleBackend(model, labels=list(bundle.get_labels()), device=device)
attacker = ASRAttacker(backend)
device

Load a 16 kHz waveform. Replace the path with your own utterance.

In [ ]:
waveform, sample_rate = torchaudio.load("example.wav")
if sample_rate != 16000:
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
print("clean:", attacker.decode(waveform))

## Untargeted FGSM

In [ ]:
adv = attacker.fgsm(waveform, epsilon=0.008, targeted=False)
print(attacker.decode(adv))
Audio(adv, rate=16000)

## Targeted BIM with early stopping

In [ ]:
target = "THE CAT SAT ON THE MAT"
adv_bim = attacker.bim(
    waveform,
    target=target,
    epsilon=0.03,
    alpha=0.004,
    num_iter=80,
    targeted=True,
    early_stop=True,
    nested=False,
)
print(attacker.decode(adv_bim))
print("WER:", attacker.wer([target], [adv_bim])[0])